# Loading Data

In [1]:
import pandas as pd
import numpy as np

In [2]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sample_submit = pd.read_csv("sample_submission.csv")

# Add Feature

In [3]:
# adding feature
def add_features(df):
    df = df.copy()
    '''df['screen_minus_sleep'] = df['daily_screen_time_hours'] - df['sleep_hours']
    df['social_ratio'] = df['social_media_hours'] / (df['daily_screen_time_hours'] + 1e-3)
    df['gaming_ratio'] = df['gaming_hours'] / (df['daily_screen_time_hours'] + 1e-3)
    df['weekend_vs_daily'] = df['weekend_screen_time'] / (df['daily_screen_time_hours'] + 1e-3)
    df['opens_per_notification'] = df['app_opens_per_day'] / (df['notifications_per_day'] + 1)
    df['leisure_hours'] = df['social_media_hours'].fillna(0) + df['gaming_hours'].fillna(0)
    df['n_missing'] = df.isna().sum(axis=1)'''
    df["social_media_hours_exp"] = df['social_media_hours']**0.3
    df["gaming_hours_exp"] = df['gaming_hours']**0.2
    df["work_study_hours_exp_015"] = df['work_study_hours']**0.15
    return df

In [4]:
# work study hours
train = add_features(train)
test = add_features(test)

# Modeling with LightGBM

In [5]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

In [6]:
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, Pool

In [7]:
# 2. Identify your string columns
string_cols = ["gender", "stress_level","academic_work_impact"]

# 3. Convert string columns to pandas 'category' type
for col in string_cols:
    train[col] = train[col].astype("category")
    test[col] = test[col].astype("category")

In [8]:
# 4. Separate features and target
X = train.drop(columns=["id","addicted_label"])
y = train["addicted_label"]
X_test = test.drop(columns=["id"])

In [9]:
SEED = 42
N_FOLDS = 5
TARGET = 'addicted_label'

In [10]:
# Stratified KFold cross validation
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(skf.split(X, y))

def run_cv(fit_predict_fn, name):
    oof = np.zeros(len(X))
    test_pred = np.zeros(len(X_test))
    scores = []
    for fold, (tr_idx, va_idx) in enumerate(FOLDS):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]
        va_pred, te_pred = fit_predict_fn(X_tr, y_tr, X_va, y_va)
        oof[va_idx] = va_pred
        test_pred += te_pred / N_FOLDS
        auc = roc_auc_score(y_va, va_pred)
        scores.append(auc)
        print(f'  fold {fold}: AUC = {auc:.5f}')
    total = roc_auc_score(y, oof)
    print(f'{name} | OOF AUC = {total:.5f} (mean {np.mean(scores):.5f} ± {np.std(scores):.5f})')
    return oof, test_pred, total

# Modeling

## Light GBM

In [11]:
# function lgb
'''
def fit_lgb(X_tr, y_tr, X_va, y_va):
    model = lgb.LGBMClassifier(
        objective='binary',
        n_estimators=5000,
        learning_rate=0.03,
        num_leaves=63,
        min_child_samples=60,
        colsample_bytree=0.8,
        subsample=0.8,
        subsample_freq=1,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=SEED,
        n_jobs=-1,
        verbose=-1,
    )
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        eval_metric='auc',
        callbacks=[lgb.early_stopping(200, verbose=False)],
        categorical_feature=string_cols,
    )
    return (model.predict_proba(X_va)[:, 1],
            model.predict_proba(X_test)[:, 1])
'''

"\ndef fit_lgb(X_tr, y_tr, X_va, y_va):\n    model = lgb.LGBMClassifier(\n        objective='binary',\n        n_estimators=5000,\n        learning_rate=0.03,\n        num_leaves=63,\n        min_child_samples=60,\n        colsample_bytree=0.8,\n        subsample=0.8,\n        subsample_freq=1,\n        reg_alpha=0.1,\n        reg_lambda=1.0,\n        random_state=SEED,\n        n_jobs=-1,\n        verbose=-1,\n    )\n    model.fit(\n        X_tr, y_tr,\n        eval_set=[(X_va, y_va)],\n        eval_metric='auc',\n        callbacks=[lgb.early_stopping(200, verbose=False)],\n        categorical_feature=string_cols,\n    )\n    return (model.predict_proba(X_va)[:, 1],\n            model.predict_proba(X_test)[:, 1])\n"

In [12]:
# oof_lgb, test_lgb, auc_lgb = run_cv(fit_lgb, 'LightGBM')

## XGBoost

In [13]:
'''
def fit_xgb(X_tr, y_tr, X_va, y_va):
    model = xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='auc',
        n_estimators=5000,
        learning_rate=0.03,
        max_depth=7,
        min_child_weight=20,
        colsample_bytree=0.8,
        subsample=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        tree_method='hist',
        enable_categorical=True,
        early_stopping_rounds=200,
        random_state=SEED,
        n_jobs=-1,
    )
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    return (model.predict_proba(X_va)[:, 1],
            model.predict_proba(X_test)[:, 1])
'''

"\ndef fit_xgb(X_tr, y_tr, X_va, y_va):\n    model = xgb.XGBClassifier(\n        objective='binary:logistic',\n        eval_metric='auc',\n        n_estimators=5000,\n        learning_rate=0.03,\n        max_depth=7,\n        min_child_weight=20,\n        colsample_bytree=0.8,\n        subsample=0.8,\n        reg_alpha=0.1,\n        reg_lambda=1.0,\n        tree_method='hist',\n        enable_categorical=True,\n        early_stopping_rounds=200,\n        random_state=SEED,\n        n_jobs=-1,\n    )\n    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)\n    return (model.predict_proba(X_va)[:, 1],\n            model.predict_proba(X_test)[:, 1])\n"

In [14]:
# oof_xgb, test_xgb, auc_xgb = run_cv(fit_xgb, 'XGBoost')

## Cat Boost

In [15]:
def cat_prepare(df):
    df = df.copy()
    for c in string_cols:
        df[c] = df[c].cat.add_categories('missing').fillna('missing').astype(str)
    return df

X_cb, X_test_cb = cat_prepare(X), cat_prepare(X_test)

def fit_cat(X_tr, y_tr, X_va, y_va):
    tr_idx, va_idx = X_tr.index, X_va.index
    X_tr, X_va = X_cb.loc[tr_idx], X_cb.loc[va_idx]
    model = CatBoostClassifier(
        iterations=5000,
        learning_rate=0.03,
        depth=7,
        l2_leaf_reg=3.0,
        eval_metric='AUC',
        cat_features=string_cols,
        random_seed=SEED,
        early_stopping_rounds=200,
        verbose=False,
    )
    model.fit(X_tr, y_tr, eval_set=(X_va, y_va))
    return (model.predict_proba(X_va)[:, 1],
            model.predict_proba(X_test_cb)[:, 1])

In [16]:
oof_cat, test_cat, auc_cat = run_cv(fit_cat, 'CatBoost')

  fold 0: AUC = 0.96257
  fold 1: AUC = 0.96299
  fold 2: AUC = 0.96335
  fold 3: AUC = 0.96399
  fold 4: AUC = 0.96300
CatBoost | OOF AUC = 0.96318 (mean 0.96318 ± 0.00047)


## Ensemble

In [17]:
'''
from scipy.stats import rankdata

# rank data used to rank the data at hand. Example, I have data of [1,4,3,5,5], the output would be [1,3,2,4,4]
# as 5 is at rank 4

results = pd.DataFrame({
    'model': ['LightGBM', 'XGBoost', 'CatBoost'],
    'oof_auc': [auc_lgb, auc_xgb, auc_cat],
}).sort_values('oof_auc', ascending=False)
display(results)

oof_mean = (oof_lgb + oof_xgb + oof_cat) / 3
oof_rank = (rankdata(oof_lgb) + rankdata(oof_xgb) + rankdata(oof_cat)) / (3 * len(oof_lgb))

print('Ensamble (media)  OOF AUC:', roc_auc_score(y, oof_mean))
print('Ensamble (rangos) OOF AUC:', roc_auc_score(y, oof_rank))
'''

"\nfrom scipy.stats import rankdata\n\n# rank data used to rank the data at hand. Example, I have data of [1,4,3,5,5], the output would be [1,3,2,4,4]\n# as 5 is at rank 4\n\nresults = pd.DataFrame({\n    'model': ['LightGBM', 'XGBoost', 'CatBoost'],\n    'oof_auc': [auc_lgb, auc_xgb, auc_cat],\n}).sort_values('oof_auc', ascending=False)\ndisplay(results)\n\noof_mean = (oof_lgb + oof_xgb + oof_cat) / 3\noof_rank = (rankdata(oof_lgb) + rankdata(oof_xgb) + rankdata(oof_cat)) / (3 * len(oof_lgb))\n\nprint('Ensamble (media)  OOF AUC:', roc_auc_score(y, oof_mean))\nprint('Ensamble (rangos) OOF AUC:', roc_auc_score(y, oof_rank))\n"

In [18]:
# test_mean = (test_lgb + test_xgb + test_cat) / 3
# test_mean = (test_lgb + test_xgb ) / 2
# test_rank = (rankdata(test_lgb) + rankdata(test_xgb) + rankdata(test_cat)) / (3 * len(test_lgb))

In [19]:
def save_sub(pred, name):
    sub = sample_submit.copy()
    sub[TARGET] = pred
    sub.to_csv(name, index=False)
    print('saving:', name)


In [20]:
#save_sub(test_lgb,  'sub_lgb_9.csv')
#save_sub(test_xgb,  'sub_xgb_9.csv')
save_sub(test_cat,  'sub_cat_10.csv')
# save_sub(test_mean, 'sub_ensemble_mean_9.csv')
# save_sub(test_rank, 'sub_ensemble_rank.csv')

saving: sub_cat_10.csv


In [21]:
def save_oof(pred, name):
    oof = train.copy()
    oof = oof[["id",TARGET]]
    oof[TARGET] = pred
    oof.to_csv(name, index=False)
    print('saving:', name)

In [22]:
# save_oof(oof_lgb,'oof_lgb_9.csv')
# save_oof(oof_xgb,'oof_xgb_9.csv')
save_oof(oof_cat,'oof_cat_10.csv')

saving: oof_cat_10.csv
